In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T
 
CONFIG = {
    "catalog": "workspace",
    "schema":  "banking_datasentry",
}
 
VALIDATED_AT = datetime.now().isoformat()
 
spark.sql(f"USE CATALOG {CONFIG['catalog']}")
spark.sql(f"USE SCHEMA {CONFIG['schema']}")
 
print(f"Catalog      : {CONFIG['catalog']}")
print(f"Schema       : {CONFIG['schema']}")
print(f"Validated at : {VALIDATED_AT}")

In [0]:
#Apply DQ Rules and Tag Records
def apply_dq_rules(df, rules: list):
    """
    Apply a list of DQ rules to a DataFrame.
    Returns the DataFrame with rule columns, dq_status, and failed_rules added.
    """
    rule_col_names = []
 
    for rule in rules:
        col_name = f"rule_{rule['name']}"
        # 1 = PASS, 0 = FAIL — using integer so we can SUM in Gold scoring
        df = df.withColumn(
            col_name,
            F.when(rule["condition"], F.lit(1)).otherwise(F.lit(0))
        )
        rule_col_names.append((col_name, rule["name"]))
 
    # dq_status: PASS only if every single rule column = 1
    all_pass_condition = F.lit(True)
    for col_name, _ in rule_col_names:
        all_pass_condition = all_pass_condition & (F.col(col_name) == 1)
 
    df = df.withColumn(
        "dq_status",
        F.when(all_pass_condition, F.lit("PASS")).otherwise(F.lit("FAIL"))
    )
 
    # failed_rules: collect names of rules where column = 0
    # Uses array_join + filter to build a readable comma-separated string
    failed_array = F.array(*[
    F.when(F.col(col_name) == 0, F.lit(rule_name))
    for col_name, rule_name in rule_col_names
    ])

    df = df.withColumn(
    "failed_rules",
    F.concat_ws(", ", *[
        F.when(F.col(col_name) == 0, F.lit(rule_name))
        for col_name, rule_name in rule_col_names
    ])
    )
 
    # Audit column
    df = df.withColumn("validated_at", F.lit(VALIDATED_AT))
 
    return df

In [0]:
#Reading Bronze Customers
bronze_customers = spark.read.format("delta").table("bronze_customers")
 
print(f"Bronze customers loaded: {bronze_customers.count():,} rows")
print(f"Columns: {bronze_customers.columns}")
bronze_customers.limit(3).show(truncate=40, vertical=True)

In [0]:
#Casting Columns for Correct Data Types
customers = bronze_customers \
    .withColumn("date_of_birth_dt",
        F.to_date(F.col("date_of_birth"), "yyyy-MM-dd")) \
    .withColumn("created_at_ts",
        F.to_timestamp(F.col("created_at")))
 
print("Type casting complete.")
print("New typed columns added: date_of_birth_dt, created_at_ts")
customers.printSchema()

## Defining DQ Rules for Customers

Here **12 DQ rules** are defined across 4 dimensions for the customers table.

Each rule is a Python dictionary with 3 keys:

| Key | Description |
|---|---|
| `name` | Snake_case identifier — appears in `failed_rules` column and Gold scorecard |
| `condition` | Spark Column expression — True means the record **PASSES** this rule |
| `dimension` | One of: Completeness, Validity, Consistency, Uniqueness |

> **Important:** The condition is always written as the **PASS case**.
> If a record does NOT satisfy the condition, it gets flagged as FAIL.

---

### Pre-work Before Rules

**Uniqueness pre-work**
We do a `groupBy` on `customer_id` to count occurrences across the whole table.
Records with `count > 1` are duplicates. This count is joined back so each row knows whether its own ID is duplicated.

---

### Rules by Dimension

**Completeness** - required fields must not be null
- `customer_id_not_null`
- `full_name_not_null`
- `email_not_null`
- `created_at_not_null`

**Validity** - values must be in correct format or range
- `email_format_valid` - must match `^[^@]+@[^@]+\.[^@]+$`
- `date_of_birth_valid` - must parse successfully as a date
- `country_code_valid` - must be exactly 2 uppercase letters
- `customer_id_uuid_format` - must match UUID pattern `8-4-4-4-12`

**Consistency** - values must be logically correct
- `created_at_not_future` - creation timestamp cannot be in the future
- `date_of_birth_not_future` - date of birth cannot be in the future
- `age_between_18_and_120` - derived age must be realistic

**Uniqueness** - no duplicate records
- `customer_id_unique` - each `customer_id` must appear exactly once

In [0]:
today = F.current_date()

# Uniqueness pre-work
# Count occurrences of each customer_id across the whole table.
# Records with count > 1 are duplicates.
# We join this back to the main DataFrame so each row knows its own count.
customer_id_counts = customers.groupBy("customer_id") \
    .agg(F.count("*").alias("customer_id_count"))

customers = customers.join(customer_id_counts, on=["customer_id"], how="left")

customer_rules = [

    # COMPLETENESS
    {"name": "customer_id_not_null",    "condition": F.col("customer_id").isNotNull(),                                           "dimension": "Completeness"},
    {"name": "full_name_not_null",      "condition": F.col("full_name").isNotNull(),                                             "dimension": "Completeness"},
    {"name": "email_not_null",          "condition": F.col("email").isNotNull(),                                                 "dimension": "Completeness"},
    {"name": "created_at_not_null",     "condition": F.col("created_at").isNotNull(),                                           "dimension": "Completeness"},

    # VALIDITY
    {"name": "email_format_valid",      "condition": F.col("email").rlike("^[^@]+@[^@]+\\.[^@]+$"),                             "dimension": "Validity"},
    {"name": "date_of_birth_valid",     "condition": F.col("date_of_birth_dt").isNotNull() | F.col("date_of_birth").isNull(),   "dimension": "Validity"},
    {"name": "country_code_valid",      "condition": F.col("country").rlike("^[A-Z]{2}$") | F.col("country").isNull(),          "dimension": "Validity"},
    {"name": "customer_id_uuid_format", "condition": F.col("customer_id").rlike("^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$"), "dimension": "Validity"},

    # CONSISTENCY
    {"name": "created_at_not_future",      "condition": F.col("created_at_ts") <= F.current_timestamp(),                                                          "dimension": "Consistency"},
    {"name": "date_of_birth_not_future",   "condition": F.col("date_of_birth_dt").isNull() | (F.col("date_of_birth_dt") < today),                                 "dimension": "Consistency"},
    {"name": "age_between_18_and_120",     "condition": F.col("date_of_birth_dt").isNull() | ((F.datediff(today, F.col("date_of_birth_dt")) / 365).between(18, 120)), "dimension": "Consistency"},

    # UNIQUENESS
    {"name": "customer_id_unique",      "condition": F.col("customer_id_count") == 1,                                           "dimension": "Uniqueness"},
]

print(f"Total rules defined for customers: {len(customer_rules)}")
for r in customer_rules:
    print(f"  [{r['dimension']:<15}] {r['name']}")

In [0]:
#Applying rules and writing silver customers
silver_customers = apply_dq_rules(customers, customer_rules)
 
# Drop the helper column used for uniqueness check — not needed in output
silver_customers = silver_customers.drop("customer_id_count")
 
# Write to Silver Delta table
silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customers")
 
print(f"silver_customers written: {silver_customers.count():,} rows")

In [0]:
#Validation Summary of Customers
print("=" * 60)
print("SILVER CUSTOMERS — VALIDATION SUMMARY")
print("=" * 60)
 
# Overall PASS/FAIL split
silver_customers.groupBy("dq_status") \
    .agg(F.count("*").alias("record_count")) \
    .withColumn("pct", F.round(F.col("record_count") / silver_customers.count() * 100, 2)) \
    .orderBy("dq_status") \
    .show()
 
# Rule-level pass rates
print("Rule-level pass rates:")
rule_cols = [c for c in silver_customers.columns if c.startswith("rule_")]
total = silver_customers.count()
 
for col in rule_cols:
    passed = silver_customers.filter(F.col(col) == 1).count()
    pct    = round(passed / total * 100, 2)
    status = "OK" if pct >= 99 else "WARN" if pct >= 95 else "FAIL"
    print(f"  [{status}] {col:<45} {passed:>7,} / {total:>7,}  ({pct}%)")
 
print("\nSample of failed records:")
silver_customers.filter(F.col("dq_status") == "FAIL") \
    .select("customer_id", "email", "date_of_birth", "dq_status", "failed_rules") \
    .limit(5) \
    .show(truncate=60)
 
print("\nNotebook 03 — Customers complete. Ready for Accounts rules.")

In [0]:
#Reading Bronze Accounts
bronze_accounts = spark.read.format("delta").table("bronze_accounts")
 
print(f"Bronze accounts loaded: {bronze_accounts.count():,} rows")
print(f"Columns: {bronze_accounts.columns}")
bronze_accounts.limit(3).show(truncate=40, vertical=True)

In [0]:
#Casting Columns to Correct Types
accounts = bronze_accounts \
    .withColumn("balance_num",
        F.col("balance").cast(T.DecimalType(18, 2))) \
    .withColumn("opened_date_dt",
        F.to_date(F.col("opened_date"), "yyyy-MM-dd"))
 
print("Type casting complete.")
print("New typed columns added: balance_num, opened_date_dt")
accounts.printSchema()

## Defining DQ Rules for Accounts

Here **13 DQ rules** are defined across 5 dimensions for the accounts table.

Each rule is a Python dictionary with 3 keys:

| Key | Description |
|---|---|
| `name` | Snake_case identifier - appears in `failed_rules` column and Gold scorecard |
| `condition` | Spark Column expression - True means the record **PASSES** this rule |
| `dimension` | One of: Completeness, Validity, Consistency, Uniqueness, Referential Integrity |

> **Important:** The condition is always written as the **PASS case**.
> If a record does NOT satisfy the condition, it gets flagged as FAIL.

---

### Pre-work Before Rules

**Uniqueness pre-work**
We do a `groupBy` on `account_id` to count occurrences across the whole table.
Records with `count > 1` are duplicates. This count is joined back so each row knows whether its own ID is duplicated.

**Referential Integrity pre-work**
We load valid `customer_id` values from `silver_customers` (PASS records only).
A left join is performed between accounts and this set.
If `matched_customer = 1` the customer exists. If null the FK is orphaned, no matching customer found.
We use **silver** (not bronze) as the reference because silver is already validated - it is the trusted source.

---

### Rules by Dimension

**Completeness** - required fields must not be null
- `account_id_not_null`
- `customer_id_not_null`
- `account_type_not_null`
- `status_not_null`

**Validity** - values must be in correct format or range
- `account_type_valid` - must be one of `CURRENT`, `SAVINGS`, `LOAN`
- `status_valid` - must be one of `ACTIVE`, `INACTIVE`, `CLOSED`
- `currency_valid` - must be a recognised 3-letter currency code
- `account_id_uuid_format` - must match UUID pattern `8-4-4-4-12`
- `opened_date_valid` - must parse successfully as a date

**Consistency** - values must be logically correct
- `savings_loan_balance_not_negative` - only `CURRENT` accounts can have negative balance (overdraft). `SAVINGS` and `LOAN` must be >= 0
- `opened_date_not_future` - an account cannot have been opened in the future

**Uniqueness** - no duplicate records
- `account_id_unique` - each `account_id` must appear exactly once

**Referential Integrity** - relationships between tables must be valid
- `customer_id_exists_in_customers` - every `customer_id` in accounts must exist as a PASS record in `silver_customers`. Orphaned foreign keys fail this rule.

In [0]:
# --- Valid value sets ---
VALID_ACCOUNT_TYPES = ["CURRENT", "SAVINGS", "LOAN"]
VALID_STATUSES      = ["ACTIVE", "INACTIVE", "CLOSED"]
VALID_CURRENCIES    = ["EUR", "GBP", "USD", "CHF", "JPY", "CAD", "AUD"]
 
today = F.current_date()
 
# --- Uniqueness pre-work ---
account_id_counts = accounts.groupBy("account_id") \
    .agg(F.count("*").alias("account_id_count"))
 
accounts = accounts.join(account_id_counts, on=["account_id"], how="left")
 
# --- Referential Integrity pre-work ---
# Load the valid customer_ids from silver_customers.
# We use silver (not bronze) because silver has already been deduplicated
# and validated — it is the trusted reference set.
valid_customer_ids = spark.read.format("delta").table("silver_customers") \
    .filter(F.col("dq_status") == "PASS") \
    .select("customer_id") \
    .distinct()
 
# Left join accounts to valid_customer_ids and flag matches
# matched_customer = 1 means the customer_id exists in silver_customers
accounts = accounts.join(
    valid_customer_ids.withColumn("matched_customer", F.lit(1)),
    on=["customer_id"],
    how="left"
)
 
# --- Rule definitions ---
account_rules = [
 
    # COMPLETENESS — required fields must not be null
    {
        "name":      "account_id_not_null",
        "condition": F.col("account_id").isNotNull(),
        "dimension": "Completeness"
    },
    {
        "name":      "customer_id_not_null",
        "condition": F.col("customer_id").isNotNull(),
        "dimension": "Completeness"
    },
    {
        "name":      "account_type_not_null",
        "condition": F.col("account_type").isNotNull(),
        "dimension": "Completeness"
    },
    {
        "name":      "status_not_null",
        "condition": F.col("status").isNotNull(),
        "dimension": "Completeness"
    },
 
    # VALIDITY — values must be in correct format or range
    {
        "name":      "account_type_valid",
        # Must be one of the 3 known account types
        "condition": F.col("account_type").isin(VALID_ACCOUNT_TYPES),
        "dimension": "Validity"
    },
    {
        "name":      "status_valid",
        # Must be one of the 3 known status values
        "condition": F.col("status").isin(VALID_STATUSES),
        "dimension": "Validity"
    },
    {
        "name":      "currency_valid",
        # Must be a recognised 3-letter currency code
        "condition": F.col("currency").isin(VALID_CURRENCIES) | F.col("currency").isNull(),
        "dimension": "Validity"
    },
    {
        "name":      "account_id_uuid_format",
        # UUID format: 8-4-4-4-12 hex characters
        "condition": F.col("account_id").rlike(
            "^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$"
        ),
        "dimension": "Validity"
    },
    {
        "name":      "opened_date_valid",
        # Cast must have succeeded — null cast means invalid date string
        "condition": F.col("opened_date_dt").isNotNull() | F.col("opened_date").isNull(),
        "dimension": "Validity"
    },
 
    # CONSISTENCY — values must be logically correct
    {
        "name":      "savings_loan_balance_not_negative",
        # SAVINGS and LOAN accounts should never have a negative balance.
        # CURRENT accounts can go negative (overdraft) — so we exclude them.
        # If balance is null we let it pass here — null check is Completeness.
        "condition": (F.col("account_type") == "CURRENT") |
                     F.col("balance_num").isNull() |
                     (F.col("balance_num") >= 0),
        "dimension": "Consistency"
    },
    {
        "name":      "opened_date_not_future",
        # An account cannot have been opened in the future
        "condition": F.col("opened_date_dt").isNull() | (F.col("opened_date_dt") <= today),
        "dimension": "Consistency"
    },
 
    # UNIQUENESS
    {
        "name":      "account_id_unique",
        # Each account_id must appear exactly once
        "condition": F.col("account_id_count") == 1,
        "dimension": "Uniqueness"
    },
 
    # REFERENTIAL INTEGRITY
    {
        "name":      "customer_id_exists_in_customers",
        # The customer_id must exist as a valid PASS record in silver_customers.
        # matched_customer = 1 means the join found a match.
        # If matched_customer is null, no match was found — orphaned FK.
        "condition": F.col("matched_customer") == 1,
        "dimension": "Referential Integrity"
    },
]
 
print(f"Total rules defined for accounts: {len(account_rules)}")
for r in account_rules:
    print(f"  [{r['dimension']:<25}] {r['name']}")


In [0]:
#Applying Rules and Write Silver Accounts 
silver_accounts = apply_dq_rules(accounts, account_rules)
 
# Drop helper columns used for rule computation - not needed in output
silver_accounts = silver_accounts.drop("account_id_count", "matched_customer")
 
# Write to Silver Delta table
silver_accounts.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_accounts")
 
print(f"silver_accounts written: {silver_accounts.count():,} rows")

In [0]:
#Validation Summary for Accounts
print("=" * 60)
print("SILVER ACCOUNTS — VALIDATION SUMMARY")
print("=" * 60)
 
# Overall PASS/FAIL split
total = silver_accounts.count()
silver_accounts.groupBy("dq_status") \
    .agg(F.count("*").alias("record_count")) \
    .withColumn("pct", F.round(F.col("record_count") / total * 100, 2)) \
    .orderBy("dq_status") \
    .show()
 
# Rule-level pass rates
print("Rule-level pass rates:")
rule_cols = [c for c in silver_accounts.columns if c.startswith("rule_")]
 
for col in rule_cols:
    passed = silver_accounts.filter(F.col(col) == 1).count()
    pct    = round(passed / total * 100, 2)
    status = "OK" if pct >= 99 else "WARN" if pct >= 95 else "FAIL"
    print(f"  [{status}] {col:<55} {passed:>7,} / {total:>7,}  ({pct}%)")
 
# Sample of failed records
print("\nSample of failed records:")
silver_accounts.filter(F.col("dq_status") == "FAIL") \
    .select("account_id", "customer_id", "account_type", "balance", "status", "dq_status", "failed_rules") \
    .limit(5) \
    .show(truncate=60)
 
print("\nNotebook 03 - Accounts complete. Ready for Transactions rules.")
 

In [0]:
#Reading Bronze Transactions
bronze_transactions = spark.read.format("delta").table("bronze_transactions")
 
print(f"Bronze transactions loaded: {bronze_transactions.count():,} rows")
print(f"Columns: {bronze_transactions.columns}")
bronze_transactions.limit(3).show(truncate=40, vertical=True)

In [0]:
#Casting Columns to Correct Types
transactions = bronze_transactions \
    .withColumn("amount_num",
        F.col("amount").cast(T.DecimalType(18, 2))) \
    .withColumn("transaction_date_ts",
        F.to_timestamp(F.col("transaction_date")))
 
print("Type casting complete.")
print("New typed columns added: amount_num, transaction_date_ts")
transactions.printSchema()

##Defining DQ Rules for Transactions

Here **16 DQ rules** are defined across 5 dimensions for the transactions table.

Each rule is a Python dictionary with 3 keys:

| Key | Description |
|---|---|
| `name` | Snake_case identifier - appears in `failed_rules` column and Gold scorecard |
| `condition` | Spark Column expression - True means the record **PASSES** this rule |
| `dimension` | One of: Completeness, Validity, Consistency, Uniqueness, Referential Integrity |

> **Important:** The condition is always written as the **PASS case**.
> If a record does NOT satisfy the condition, it gets flagged as FAIL.

---

### Pre-work Before Rules

**Uniqueness pre-work**
We do a `groupBy` on `transaction_id` to count occurrences across the whole table.
Records with `count > 1` are duplicates. This count is joined back so each row knows whether its own ID is duplicated.

**Referential Integrity pre-work**
We load valid `account_id` values from `silver_accounts` (PASS records only).
A left join is performed between transactions and this set.
If `matched_account = 1` the account exists. If null the FK is orphaned, no matching account found.
We use **silver** (not bronze) as the reference because silver is already validated - it is the trusted source.

---

### Rules by Dimension

**Completeness** - required fields must not be null
- `transaction_id_not_null`
- `account_id_not_null`
- `transaction_type_not_null`
- `amount_not_null`
- `currency_not_null`
- `transaction_date_not_null`
- `status_not_null`

**Validity** - values must be in correct format or range
- `transaction_type_valid` - must be one of `DEBIT`, `CREDIT`, `TRANSFER`
- `status_valid` - must be one of `COMPLETED`, `PENDING`, `FAILED`, `REVERSED`
- `currency_valid` - must be a recognised 3-letter currency code
- `transaction_id_uuid_format` - must match UUID pattern `8-4-4-4-12`
- `amount_greater_than_zero` - zero-amount transactions are invalid in financial data

**Consistency** - values must be logically correct
- `transaction_date_not_future` - a transaction cannot have occurred in the future
- `failed_reversed_amount_not_positive` - `FAILED` and `REVERSED` transactions represent rejected or rolled-back events. Their amount should never be positive. This is a business logic rule - not just a format check.

**Uniqueness** - no duplicate records
- `transaction_id_unique` - each `transaction_id` must appear exactly once

**Referential Integrity** - relationships between tables must be valid
- `account_id_exists_in_accounts` - every `account_id` in transactions must exist as a PASS record in `silver_accounts`. Orphaned foreign keys fail this rule.

In [0]:
# --- Valid value sets ---
VALID_TXN_TYPES  = ["DEBIT", "CREDIT", "TRANSFER"]
VALID_TXN_STATUS = ["COMPLETED", "PENDING", "FAILED", "REVERSED"]
VALID_CURRENCIES = ["EUR", "GBP", "USD", "CHF", "JPY", "CAD", "AUD"]
 
today = F.current_date()
 
# --- Uniqueness pre-work ---
# Count occurrences of each transaction_id across the whole table.
# Records with count > 1 are duplicates.
txn_id_counts = transactions.groupBy("transaction_id") \
    .agg(F.count("*").alias("txn_id_count"))
 
transactions = transactions.join(txn_id_counts, on=["transaction_id"], how="left")
 
# --- Referential Integrity pre-work ---
# Load valid account_ids from silver_accounts (PASS records only).
# A left join flags whether each transaction's account_id exists in silver.
# matched_account = 1 means the account exists. null means orphaned FK.
valid_account_ids = spark.read.format("delta").table("silver_accounts") \
    .filter(F.col("dq_status") == "PASS") \
    .select("account_id") \
    .distinct()
 
transactions = transactions.join(
    valid_account_ids.withColumn("matched_account", F.lit(1)),
    on=["account_id"],
    how="left"
)
 
# --- Rule definitions ---
transaction_rules = [
 
    # COMPLETENESS — required fields must not be null
    {
        "name":      "transaction_id_not_null",
        "condition": F.col("transaction_id").isNotNull(),
        "dimension": "Completeness"
    },
    {
        "name":      "account_id_not_null",
        "condition": F.col("account_id").isNotNull(),
        "dimension": "Completeness"
    },
    {
        "name":      "transaction_type_not_null",
        "condition": F.col("transaction_type").isNotNull(),
        "dimension": "Completeness"
    },
    {
        "name":      "amount_not_null",
        "condition": F.col("amount").isNotNull(),
        "dimension": "Completeness"
    },
    {
        "name":      "currency_not_null",
        "condition": F.col("currency").isNotNull(),
        "dimension": "Completeness"
    },
    {
        "name":      "transaction_date_not_null",
        "condition": F.col("transaction_date").isNotNull(),
        "dimension": "Completeness"
    },
    {
        "name":      "status_not_null",
        "condition": F.col("status").isNotNull(),
        "dimension": "Completeness"
    },
 
    # VALIDITY — values must be in correct format or range
    {
        "name":      "transaction_type_valid",
        # Must be one of the 3 known transaction types
        "condition": F.col("transaction_type").isin(VALID_TXN_TYPES),
        "dimension": "Validity"
    },
    {
        "name":      "status_valid",
        # Must be one of the 4 known status values
        "condition": F.col("status").isin(VALID_TXN_STATUS),
        "dimension": "Validity"
    },
    {
        "name":      "currency_valid",
        # Must be a recognised 3-letter currency code
        "condition": F.col("currency").isin(VALID_CURRENCIES) | F.col("currency").isNull(),
        "dimension": "Validity"
    },
    {
        "name":      "transaction_id_uuid_format",
        # UUID format: 8-4-4-4-12 hex characters
        "condition": F.col("transaction_id").rlike(
            "^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$"
        ),
        "dimension": "Validity"
    },
    {
        "name":      "amount_greater_than_zero",
        # Zero-amount transactions are invalid in financial data.
        # Null amounts are handled by the completeness rule above.
        "condition": F.col("amount_num").isNull() | (F.col("amount_num") > 0),
        "dimension": "Validity"
    },
 
    # CONSISTENCY — values must be logically correct
    {
        "name":      "transaction_date_not_future",
        # A transaction cannot have occurred in the future
        "condition": F.col("transaction_date_ts").isNull() |
                     (F.col("transaction_date_ts") <= F.current_timestamp()),
        "dimension": "Consistency"
    },
    {
        "name":      "failed_reversed_amount_not_positive",
        # FAILED and REVERSED transactions should not have a positive amount.
        # In real financial systems, these represent rejected or rolled-back
        # transactions — their amount should be 0 or negative.
        # We allow null amounts to pass here — completeness rule handles nulls.
        "condition": (~F.col("status").isin(["FAILED", "REVERSED"])) |
                     F.col("amount_num").isNull() |
                     (F.col("amount_num") <= 0),
        "dimension": "Consistency"
    },
 
    # UNIQUENESS
    {
        "name":      "transaction_id_unique",
        # Each transaction_id must appear exactly once
        "condition": F.col("txn_id_count") == 1,
        "dimension": "Uniqueness"
    },
 
    # REFERENTIAL INTEGRITY
    {
        "name":      "account_id_exists_in_accounts",
        # The account_id must exist as a PASS record in silver_accounts.
        # matched_account = 1 means the join found a match.
        # null means no match — orphaned FK.
        "condition": F.col("matched_account") == 1,
        "dimension": "Referential Integrity"
    },
]
 
print(f"Total rules defined for transactions: {len(transaction_rules)}")
for r in transaction_rules:
    print(f"  [{r['dimension']:<25}] {r['name']}")

In [0]:
#Applying Rules
silver_transactions = apply_dq_rules(transactions, transaction_rules)
 
# Drop helper columns used for rule computation
silver_transactions = silver_transactions.drop("txn_id_count", "matched_account")
 
# Write to Silver Delta table
silver_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_transactions")
 
print(f"silver_transactions written: {silver_transactions.count():,} rows")

In [0]:
#Validation Summary of transactions silver
print("=" * 60)
print("SILVER TRANSACTIONS — VALIDATION SUMMARY")
print("=" * 60)
 
# Overall PASS/FAIL split
total = silver_transactions.count()
silver_transactions.groupBy("dq_status") \
    .agg(F.count("*").alias("record_count")) \
    .withColumn("pct", F.round(F.col("record_count") / total * 100, 2)) \
    .orderBy("dq_status") \
    .show()
 
# Rule-level pass rates
print("Rule-level pass rates:")
rule_cols = [c for c in silver_transactions.columns if c.startswith("rule_")]
 
for col in rule_cols:
    passed = silver_transactions.filter(F.col(col) == 1).count()
    pct    = round(passed / total * 100, 2)
    status = "OK" if pct >= 99 else "WARN" if pct >= 95 else "FAIL"
    print(f"  [{status}] {col:<60} {passed:>8,} / {total:>8,}  ({pct}%)")
 
# Sample of failed records
print("\nSample of failed records:")
silver_transactions.filter(F.col("dq_status") == "FAIL") \
    .select("transaction_id", "account_id", "amount", "currency",
            "transaction_date", "status", "dq_status", "failed_rules") \
    .limit(5) \
    .show(truncate=60)
 
print("\nNotebook 03 — Transactions complete.")
print("All 3 Silver tables ready. Proceed to Notebook 04 — Gold Scoring.")